# WNBA Draft Fit Predictor — Data Cleaning & EDA

This notebook cleans and explores three datasets:
1. NCAA prospect career stats (2026 draft class, Round 1)
2. WNBA rookie season stats (2019–2025 draft classes)
3. WNBA team stats (2019–2025)

Goal: Prepare clean, merged datasets ready for feature engineering.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

print("All imports successful!")

All imports successful!


In [4]:
import os

prospect_files = [
    ("Azzi Fudd", "azzi_fudd.csv"),
    ("Olivia Miles", "olivia_miles.csv"),
    ("Lauren Betts", "lauren_betts.csv"),
    ("Gabriela Jaquez", "gabriela_jaquez.csv"),
    ("Kiki Rice", "kiki_rice.csv"),
    ("Flau'jae Johnson", "flauajae_johnson.csv"),
    ("Angela Dugalic", "angela_dugalic.csv"),
    ("Raven Johnson", "raven_johnson.csv"),
    ("Cotie McMahon", "cotie_mcmahon.csv"),
    ("Madina Okot", "madina_okot.csv"),
    ("Taina Mair", "taina_mair.csv"),
    ("Gianna Kneepkens", "gianna_kneepkens.csv"),
]

all_prospects = []

for name, filename in prospect_files:
    filepath = f"../data/external/{filename}"
    try:
        df = pd.read_csv(filepath)
        df["player"] = name
        all_prospects.append(df)
        print(f"✓ {name}")
    except Exception as e:
        print(f"✗ {name}: {e}")

prospects_df = pd.concat(all_prospects, ignore_index=True)
print(f"\nTotal rows: {prospects_df.shape[0]}")
print(f"Total columns: {prospects_df.shape[1]}")

✓ Azzi Fudd
✓ Olivia Miles
✓ Lauren Betts
✓ Gabriela Jaquez
✓ Kiki Rice
✓ Flau'jae Johnson
✓ Angela Dugalic
✓ Raven Johnson
✓ Cotie McMahon
✓ Madina Okot
✓ Taina Mair
✓ Gianna Kneepkens

Total rows: 89
Total columns: 32


In [5]:
# Look at the first few rows
print(prospects_df.head())
print("\n")

# Check for missing values
print("Missing values per column:")
print(prospects_df.isnull().sum())

    Season   Team      Conf Class  Pos     G    GS    MP   FG   FGA  ...  DRB  \
0      NaN    NaN       NaN   NaN  NaN   NaN   NaN   NaN  NaN   NaN  ...  NaN   
1  2021-22  UConn  Big East    FR    G  25.0  17.0  28.0  4.3   9.4  ...  2.4   
2  2022-23  UConn  Big East    SO    G  15.0  10.0  28.2  5.9  13.0  ...  1.5   
3  2023-24  UConn  Big East    JR    G   2.0   2.0  30.5  4.0  12.5  ...  1.5   
4  2024-25  UConn  Big East    SR    G  34.0  30.0  26.5  5.1  10.9  ...  1.4   

   TRB  AST  STL  BLK  TOV   PF   PTS Awards     player  
0  NaN  NaN  NaN  NaN  NaN  NaN   NaN    NaN  Azzi Fudd  
1  2.7  1.0  1.0  0.7  1.0  1.0  12.1    NaN  Azzi Fudd  
2  1.9  1.9  1.3  0.3  1.7  1.0  15.1    NaN  Azzi Fudd  
3  2.5  2.5  1.0  0.0  1.5  2.5  11.0    NaN  Azzi Fudd  
4  2.0  1.8  1.4  0.3  1.0  1.1  13.6    NaN  Azzi Fudd  

[5 rows x 32 columns]


Missing values per column:
Season     8
Team       8
Conf       8
Class      8
Pos       34
G          8
GS         8
MP         8
FG       

In [6]:
# Drop blank rows (where Season is NaN)
prospects_df = prospects_df[prospects_df['Season'].notna()]

# Separate career totals from season rows
career_df = prospects_df[prospects_df['Season'] == 'Career'].copy()
prospects_df = prospects_df[prospects_df['Season'] != 'Career'].copy()

# Drop Awards column - not useful for modeling
prospects_df = prospects_df.drop(columns=['Awards'])

# Reset index
prospects_df = prospects_df.reset_index(drop=True)

print(f"Season rows: {prospects_df.shape[0]}")
print(f"Career rows: {career_df.shape[0]}")
print(prospects_df.head())

Season rows: 69
Career rows: 12
    Season   Team      Conf Class Pos     G    GS    MP   FG   FGA  ...  ORB  \
0  2021-22  UConn  Big East    FR   G  25.0  17.0  28.0  4.3   9.4  ...  0.3   
1  2022-23  UConn  Big East    SO   G  15.0  10.0  28.2  5.9  13.0  ...  0.4   
2  2023-24  UConn  Big East    JR   G   2.0   2.0  30.5  4.0  12.5  ...  1.0   
3  2024-25  UConn  Big East    SR   G  34.0  30.0  26.5  5.1  10.9  ...  0.6   
4  2025-26  UConn  Big East    SR   G  39.0  39.0  28.7  6.6  13.7  ...  0.7   

   DRB  TRB  AST  STL  BLK  TOV   PF   PTS     player  
0  2.4  2.7  1.0  1.0  0.7  1.0  1.0  12.1  Azzi Fudd  
1  1.5  1.9  1.9  1.3  0.3  1.7  1.0  15.1  Azzi Fudd  
2  1.5  2.5  2.5  1.0  0.0  1.5  2.5  11.0  Azzi Fudd  
3  1.4  2.0  1.8  1.4  0.3  1.0  1.1  13.6  Azzi Fudd  
4  1.8  2.6  3.1  2.5  0.5  1.5  0.8  17.3  Azzi Fudd  

[5 rows x 31 columns]


In [7]:
# Convert numeric columns to float
numeric_cols = ['G', 'GS', 'MP', 'FG', 'FGA', 'FG%', '3P', '3PA', '3P%',
                '2P', '2PA', '2P%', 'eFG%', 'FT', 'FTA', 'FT%', 'ORB',
                'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS']

for col in numeric_cols:
    prospects_df[col] = pd.to_numeric(prospects_df[col], errors='coerce')

# Check missing values now
print("Missing values after conversion:")
print(prospects_df[numeric_cols].isnull().sum())

Missing values after conversion:
G        4
GS       4
MP       4
FG       4
FGA      4
FG%      4
3P       4
3PA      4
3P%     11
2P       4
2PA      4
2P%      4
eFG%     4
FT       4
FTA      4
FT%      5
ORB      4
DRB      4
TRB      4
AST      4
STL      4
BLK      4
TOV      4
PF       4
PTS      4
dtype: int64


In [8]:
# Find rows with missing values
missing_rows = prospects_df[prospects_df[numeric_cols].isnull().any(axis=1)]
print(missing_rows[['player', 'Season', 'G', 'MP', 'PTS', '3P%', 'FT%']])

            player                    Season     G    MP   PTS  3P%    FT%
8     Olivia Miles                   2023-24   NaN   NaN   NaN  NaN    NaN
13    Lauren Betts                   2022-23  33.0   9.6   5.9  NaN  0.567
14    Lauren Betts                   2023-24  29.0  27.2  14.9  NaN  0.610
15    Lauren Betts                   2024-25  34.0  30.1  20.2  NaN  0.620
17    Lauren Betts           Stanford (1 Yr)  33.0   9.6   5.9  NaN  0.567
33  Angela Dugalic                   2022-23   NaN   NaN   NaN  NaN    NaN
39   Raven Johnson                   2021-22   2.0   4.0   0.0  NaN    NaN
50     Madina Okot                   2022-23   NaN   NaN   NaN  NaN    NaN
51     Madina Okot                   2023-24   NaN   NaN   NaN  NaN    NaN
52     Madina Okot                   2024-25  34.0  22.6  11.3  NaN  0.625
54     Madina Okot  Mississippi State (1 Yr)  34.0  22.6  11.3  NaN  0.625


In [9]:
# Drop rows where G is NaN (player didn't play that season)
prospects_df = prospects_df[prospects_df['G'].notna()].copy()

# Drop transfer notation duplicate rows
prospects_df = prospects_df[~prospects_df['Season'].str.contains('Yr', na=False)].copy()

# Fill missing 3P% and FT% with 0 (no attempts)
prospects_df['3P%'] = prospects_df['3P%'].fillna(0)
prospects_df['FT%'] = prospects_df['FT%'].fillna(0)

# Reset index
prospects_df = prospects_df.reset_index(drop=True)

print(f"Rows remaining: {prospects_df.shape[0]}")
print("Missing values remaining:")
print(prospects_df[numeric_cols].isnull().sum())

Rows remaining: 51
Missing values remaining:
G       0
GS      0
MP      0
FG      0
FGA     0
FG%     0
3P      0
3PA     0
3P%     0
2P      0
2PA     0
2P%     0
eFG%    0
FT      0
FTA     0
FT%     0
ORB     0
DRB     0
TRB     0
AST     0
STL     0
BLK     0
TOV     0
PF      0
PTS     0
dtype: int64


In [10]:
# Save cleaned prospects
prospects_df.to_csv("../data/processed/prospects_2026_clean.csv", index=False)
print("Saved prospects_2026_clean.csv!")

# Quick summary
print(f"\nPlayers: {prospects_df['player'].nunique()}")
print(f"Seasons per player:")
print(prospects_df.groupby('player')['Season'].count())

Saved prospects_2026_clean.csv!

Players: 12
Seasons per player:
player
Angela Dugalic      5
Azzi Fudd           5
Cotie McMahon       4
Flau'jae Johnson    4
Gabriela Jaquez     4
Gianna Kneepkens    5
Kiki Rice           4
Lauren Betts        4
Madina Okot         2
Olivia Miles        5
Raven Johnson       5
Taina Mair          4
Name: Season, dtype: int64


In [11]:
# Load all rookie CSVs
rookie_files = [
    (2019, "wnba_rookies_2019.csv"),
    (2020, "wnba_rookies_2020.csv"),
    (2021, "wnba_rookies_2021.csv"),
    (2022, "wnba_rookies_2022.csv"),
    (2023, "wnba_rookies_2023.csv"),
    (2024, "wnba_rookies_2024.csv"),
    (2025, "wnba_rookies_2025.csv"),
]

all_rookies = []

for year, filename in rookie_files:
    filepath = f"../data/external/{filename}"
    try:
        df = pd.read_csv(filepath)
        df["draft_year"] = year
        all_rookies.append(df)
        print(f"✓ {year} — {len(df)} players")
    except Exception as e:
        print(f"✗ {year}: {e}")

rookies_df = pd.concat(all_rookies, ignore_index=True)
print(f"\nTotal rows: {rookies_df.shape[0]}")
print(f"Total columns: {rookies_df.shape[1]}")

✓ 2019 — 37 players
✓ 2020 — 37 players
✓ 2021 — 37 players
✓ 2022 — 37 players
✓ 2023 — 37 players
✓ 2024 — 37 players
✓ 2025 — 39 players

Total rows: 261
Total columns: 16


In [12]:
print(rookies_df.head(10))
print("\nColumns:", rookies_df.columns.tolist())
print("\nMissing values:")
print(rookies_df.isnull().sum())

  Unnamed: 0          Unnamed: 1           Unnamed: 2 Unnamed: 3  \
0         Pk                Team               Player     Former   
1          1      Las Vegas Aces         Jackie Young        NaN   
2          2    New York Liberty              AD Durr        NaN   
3          3       Indiana Fever       Teaira McCowan        NaN   
4          4         Chicago Sky  Katie Lou Samuelson        NaN   
5          5        Dallas Wings     Arike Ogunbowale        NaN   
6          6      Minnesota Lynx     Napheesa Collier        NaN   
7          7  Los Angeles Sparks         Kalani Brown        NaN   
8          8     Phoenix Mercury         Alanna Smith        NaN   
9          9     Connecticut Sun      Kristine Anigwe        NaN   

          Unnamed: 4 Unnamed: 5 Unnamed: 6  Unnamed: 7 Per Game Per Game.1  \
0            College        Yrs          G         NaN       MP        PTS   
1         Notre Dame          7        243         NaN     30.0       14.0   
2         Louisvi

In [13]:
# Reload with proper headers
all_rookies = []

for year, filename in rookie_files:
    filepath = f"../data/external/{filename}"
    try:
        df = pd.read_csv(filepath, header=1)  # skip the first header row
        df["draft_year"] = year
        all_rookies.append(df)
    except Exception as e:
        print(f"✗ {year}: {e}")

rookies_df = pd.concat(all_rookies, ignore_index=True)
print(rookies_df.head())
print("\nColumns:", rookies_df.columns.tolist())

   Pk              Team               Player Former            College  Yrs  \
0   1    Las Vegas Aces         Jackie Young    NaN         Notre Dame    7   
1   2  New York Liberty              AD Durr    NaN         Louisville    3   
2   3     Indiana Fever       Teaira McCowan    NaN  Mississippi State    7   
3   4       Chicago Sky  Katie Lou Samuelson    NaN              UConn    5   
4   5      Dallas Wings     Arike Ogunbowale    NaN         Notre Dame    7   

       G  Unnamed: 7    MP   PTS  TRB  AST  Unnamed: 12    WS  WS/40  \
0  243.0         NaN  30.0  14.0  4.1  4.2          NaN  31.0  0.170   
1   79.0         NaN  15.7   6.6  1.2  1.1          NaN   0.9  0.027   
2  207.0         NaN  22.6  10.7  8.1  1.0          NaN  17.7  0.152   
3  138.0         NaN  20.1   5.9  2.5  1.4          NaN   3.5  0.050   
4  224.0         NaN  34.2  19.9  3.2  4.0          NaN  18.7  0.097   

   draft_year  
0        2019  
1        2019  
2        2019  
3        2019  
4        201

In [14]:
# Drop unnamed columns
rookies_df = rookies_df.drop(columns=['Unnamed: 7', 'Unnamed: 12', 'Former'])

# Rename columns for clarity
rookies_df = rookies_df.rename(columns={
    'Pk': 'draft_pick',
    'Team': 'wnba_team',
    'Player': 'player',
    'College': 'college',
    'Yrs': 'wnba_years',
    'G': 'wnba_games',
    'MP': 'wnba_mpg',
    'PTS': 'wnba_ppg',
    'TRB': 'wnba_rpg',
    'AST': 'wnba_apg',
    'WS': 'wnba_ws',
    'WS/40': 'wnba_ws40'
})

# Drop rows where player is NaN
rookies_df = rookies_df[rookies_df['player'].notna()].copy()

# Reset index
rookies_df = rookies_df.reset_index(drop=True)

print(f"Rows: {rookies_df.shape[0]}")
print(rookies_df.head())

Rows: 254
   draft_pick         wnba_team               player            college  \
0           1    Las Vegas Aces         Jackie Young         Notre Dame   
1           2  New York Liberty              AD Durr         Louisville   
2           3     Indiana Fever       Teaira McCowan  Mississippi State   
3           4       Chicago Sky  Katie Lou Samuelson              UConn   
4           5      Dallas Wings     Arike Ogunbowale         Notre Dame   

   wnba_years  wnba_games  wnba_mpg  wnba_ppg  wnba_rpg  wnba_apg  wnba_ws  \
0           7       243.0      30.0      14.0       4.1       4.2     31.0   
1           3        79.0      15.7       6.6       1.2       1.1      0.9   
2           7       207.0      22.6      10.7       8.1       1.0     17.7   
3           5       138.0      20.1       5.9       2.5       1.4      3.5   
4           7       224.0      34.2      19.9       3.2       4.0     18.7   

   wnba_ws40  draft_year  
0      0.170        2019  
1      0.027    

In [15]:
print("Missing values:")
print(rookies_df.isnull().sum())

# How many players never played (wnba_games is NaN or 0)?
never_played = rookies_df[rookies_df['wnba_games'].isna() | (rookies_df['wnba_games'] == 0)]
print(f"\nPlayers who never played: {len(never_played)}")
print(never_played[['player', 'wnba_team', 'draft_year', 'wnba_games']])

Missing values:
draft_pick     0
wnba_team      0
player         0
college       31
wnba_years     0
wnba_games    85
wnba_mpg      85
wnba_ppg      85
wnba_rpg      85
wnba_apg      85
wnba_ws       85
wnba_ws40     85
draft_year     0
dtype: int64

Players who never played: 85
                player           wnba_team  draft_year  wnba_games
19      Cierra Dillard      Minnesota Lynx        2019         NaN
26         Maria Conde         Chicago Sky        2019         NaN
27     Caliya Robinson       Indiana Fever        2019         NaN
30   Ángela Salvadores  Los Angeles Sparks        2019         NaN
32      Regan Magarity     Connecticut Sun        2019         NaN
..                 ...                 ...         ...         ...
248        Yvonne Ejim       Indiana Fever        2025         NaN
249       Jordan Hobbs       Seattle Storm        2025         NaN
250     Harmoni Turner      Las Vegas Aces        2025         NaN
252     Aubrey Griffin      Minnesota Lynx        

In [16]:
# Keep only Round 1 picks (picks 1-15)
rookies_df = rookies_df[pd.to_numeric(rookies_df['draft_pick'], errors='coerce') <= 15].copy()

# Reset index
rookies_df = rookies_df.reset_index(drop=True)

print(f"Rows after Round 1 filter: {rookies_df.shape[0]}")
print(f"\nPlayers who never played:")
never_played = rookies_df[rookies_df['wnba_games'].isna()]
print(f"{len(never_played)} players")
print(never_played[['player', 'wnba_team', 'draft_year', 'draft_pick']])

Rows after Round 1 filter: 105

Players who never played:
9 players
                     player               wnba_team  draft_year  draft_pick
44  Raquel Carrera Quintana           Atlanta Dream        2021          15
52          Mya Hollingshed          Minnesota Lynx        2022           8
71              Maia Hirsch          Minnesota Lynx        2023          12
73           Shaneice Swain      Los Angeles Sparks        2023          14
86            Nyadiew Puoch           Atlanta Dream        2024          12
87           Brynna Maxwell             Chicago Sky        2024          13
94             Justė Jocytė  Golden State Valkyries        2025           5
95           Georgia Amoore      Washington Mystics        2025           6
99               Ajša Sivka             Chicago Sky        2025          10


In [17]:
# Keep only players who actually played
rookies_df = rookies_df[rookies_df['wnba_games'].notna()].copy()

# Convert numeric columns
numeric_cols = ['draft_pick', 'wnba_games', 'wnba_mpg', 'wnba_ppg', 
                'wnba_rpg', 'wnba_apg', 'wnba_ws', 'wnba_ws40']

for col in numeric_cols:
    rookies_df[col] = pd.to_numeric(rookies_df[col], errors='coerce')

rookies_df = rookies_df.reset_index(drop=True)

print(f"Final rookie rows: {rookies_df.shape[0]}")
print(f"\nMissing values:")
print(rookies_df[numeric_cols].isnull().sum())

Final rookie rows: 96

Missing values:
draft_pick    0
wnba_games    0
wnba_mpg      0
wnba_ppg      0
wnba_rpg      0
wnba_apg      0
wnba_ws       0
wnba_ws40     0
dtype: int64


In [18]:
# Save cleaned rookies
rookies_df.to_csv("../data/processed/rookies_clean.csv", index=False)
print("Saved rookies_clean.csv!")

# Quick summary stats
print(f"\nRookies per draft year:")
print(rookies_df.groupby('draft_year')['player'].count())

print(f"\nAverage rookie stats:")
print(rookies_df[['wnba_ppg', 'wnba_rpg', 'wnba_apg', 'wnba_mpg', 'wnba_ws40']].describe().round(2))

Saved rookies_clean.csv!

Rookies per draft year:
draft_year
2019    15
2020    15
2021    14
2022    14
2023    13
2024    13
2025    12
Name: player, dtype: int64

Average rookie stats:
       wnba_ppg  wnba_rpg  wnba_apg  wnba_mpg  wnba_ws40
count     96.00     96.00     96.00     96.00      96.00
mean       6.32      2.97      1.40     16.60       0.03
std        4.88      2.32      1.37      8.43       0.10
min        0.10      0.10      0.00      3.60      -0.41
25%        2.75      1.20      0.50     10.20       0.00
50%        5.25      2.50      1.00     15.75       0.04
75%        7.75      3.92      1.75     22.32       0.10
max       19.90     12.90      8.50     34.30       0.23


In [19]:
# Load team stats
team_files = [
    (2019, "wnba_team_stats_2019.csv"),
    (2020, "wnba_team_stats_2020.csv"),
    (2021, "wnba_team_stats_2021.csv"),
    (2022, "wnba_team_stats_2022.csv"),
    (2023, "wnba_team_stats_2023.csv"),
    (2024, "wnba_team_stats_2024.csv"),
    (2025, "wnba_team_stats_2025.csv"),
]

all_teams = []

for year, filename in team_files:
    filepath = f"../data/external/{filename}"
    try:
        df = pd.read_csv(filepath)
        df["season"] = year
        all_teams.append(df)
        print(f"✓ {year}")
    except Exception as e:
        print(f"✗ {year}: {e}")

teams_df = pd.concat(all_teams, ignore_index=True)
print(f"\nTotal rows: {teams_df.shape[0]}")
print(teams_df.head())

✓ 2019
✓ 2020
✓ 2021
✓ 2022
✓ 2023
✓ 2024
✓ 2025

Total rows: 92
    Rk                 Team   G     MP    FG   FGA    FG%   3P   3PA    3P%  \
0  1.0  Washington Mystics*  34  200.7  32.8  69.9  0.469  9.3  25.4  0.366   
1  2.0         Chicago Sky*  34  200.7  31.4  70.0  0.448  7.3  21.6  0.336   
2  3.0      Las Vegas Aces*  34  201.5  30.0  70.3  0.427  5.5  14.9  0.368   
3  4.0     Connecticut Sun*  34  200.7  30.2  71.3  0.423  7.5  21.0  0.356   
4  5.0  Los Angeles Sparks*  34  200.7  30.3  70.2  0.432  7.1  20.9  0.341   

   ...   ORB   DRB   TRB   AST  STL  BLK   TOV    PF   PTS  season  
0  ...   8.4  24.9  33.3  21.9  7.7  4.6  10.9  15.5  89.3    2019  
1  ...   8.2  28.2  36.4  21.6  6.8  4.5  14.4  17.7  84.6    2019  
2  ...   9.2  29.7  38.9  20.9  7.0  4.6  14.1  16.9  82.2    2019  
3  ...  10.9  25.9  36.8  19.2  8.9  3.9  13.2  17.4  80.8    2019  
4  ...   8.6  25.6  34.2  18.6  8.2  3.8  13.5  18.1  80.1    2019  

[5 rows x 26 columns]


In [20]:
# Drop Rk column, clean team names (remove * for playoff teams)
teams_df = teams_df.drop(columns=['Rk'])
teams_df['Team'] = teams_df['Team'].str.replace('*', '', regex=False)

# Drop league average row
teams_df = teams_df[teams_df['Team'] != 'League Average'].copy()

# Reset index
teams_df = teams_df.reset_index(drop=True)

# Save
teams_df.to_csv("../data/processed/team_stats_clean.csv", index=False)
print("Saved team_stats_clean.csv!")
print(f"Rows: {teams_df.shape[0]}")
print(f"Teams per year: {teams_df.groupby('season')['Team'].count()}")

Saved team_stats_clean.csv!
Rows: 85
Teams per year: season
2019    12
2020    12
2021    12
2022    12
2023    12
2024    12
2025    13
Name: Team, dtype: int64
